# 필요한 것만 꺼내기

> 파이썬 2강 · 데이터 다루기

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [필요한 것만 꺼내기](https://mioon1402.github.io/timeseriesdata/python/p02-select-filter.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

print('준비 완료')

## 1. 열 꺼내기

**2-1. 대괄호 한 겹 vs 두 겹**

In [ ]:
import pandas as pd
df = pd.read_csv("cafe_sales.csv")

한겹 = df["sales"]                       # Series (1차원)
두겹 = df[["date", "weekday", "sales"]]  # DataFrame (2차원 표)

print("한 겹:", type(한겹).__name__)
print("두 겹:", type(두겹).__name__)
print()
print(한겹.head(3))

**2-2. 여러 열 골라 보기**

In [ ]:
df[["date", "weekday", "sales"]].head()

## 2. 조건으로 행 고르기

**2-3. 조건은 True/False 목록을 만든다**

In [ ]:
조건 = df["visitors"] >= 250

print(조건.head())
print()
print("전체 길이:", len(조건))
print("True 개수:", 조건.sum())

**2-4. 체를 씌워 걸러내기**

In [ ]:
busy = df[df["visitors"] >= 250]

print("250명 이상인 날:", len(busy), "일")
busy[["date", "weekday", "visitors", "sales"]].head()

## 3. 조건 여러 개 붙이기

**2-5. 비 온 주말 찾기**

In [ ]:
비온주말 = df[(df["weekday"].isin(["토", "일"])) & (df["rain_mm"] > 0)]
print("비 온 주말:", len(비온주말), "일")
print(비온주말[["date", "weekday", "rain_mm", "visitors"]].head(3))

# 또는(OR) — 무더위였거나 공휴일이었던 날
특별한날 = df[(df["avg_temp"] > 30) | (df["is_holiday"])]
print()
print("무더위 또는 공휴일:", len(특별한날), "일")

# 아닌 것(NOT) — 공휴일이 아닌 날
print("공휴일이 아닌 날:", len(df[~df["is_holiday"]]), "일")

## 4. 자주 쓰는 조건 도구

**2-6. isin · between · 문자열**

In [ ]:
# isin — 목록 안에 있는가
봄 = df[df["date"].str[5:7].isin(["03", "04", "05"])]
print("3~5월:", len(봄), "일")

# between — 범위 안에 있는가 (양쪽 끝 포함)
쾌적 = df[df["avg_temp"].between(20, 25)]
print("기온 20~25도:", len(쾌적), "일")

# 문자열 다루기 — .str 을 붙이면 글자로 다룰 수 있다
print("2025년 데이터:", len(df[df["date"].str.startswith("2025")]), "일")

## 5. 정렬하기

**2-7. sort_values**

In [ ]:
# 매출 높은 순
print(df.sort_values("sales", ascending=False)[["date", "weekday", "sales"]].head(3))

# 두 기준으로 — 요일 순서대로, 각 요일 안에서는 매출 높은 순
print()
print(df.sort_values(["weekday", "sales"], ascending=[True, False])
        [["weekday", "date", "sales"]].head(3))

## 6. loc과 iloc

**2-8. loc — 이름으로 고르기**

In [ ]:
# df.loc[행조건, 열목록]
df.loc[df["visitors"] > 300, ["date", "weekday", "visitors", "sales"]]

**2-9. iloc — 위치 번호로 고르기**

In [ ]:
# 앞에서 3행, 앞에서 3열
df.iloc[:3, :3]

## 7. 실전: 비가 매출에 영향을 줄까?

**2-10. 비 온 날 vs 안 온 날**

In [ ]:
비온날 = df[df["rain_mm"] > 0]["visitors"]
맑은날 = df[df["rain_mm"] == 0]["visitors"]

print(f"비 온 날    {len(비온날):3d}일   평균 {비온날.mean():.1f}명")
print(f"비 안 온 날 {len(맑은날):3d}일   평균 {맑은날.mean():.1f}명")
print()
print(f"차이 {맑은날.mean() - 비온날.mean():.1f}명 "
      f"({(비온날.mean() / 맑은날.mean() - 1) * 100:+.1f}%)")

## 8. 직접 해보기

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 주말(토·일) 중 매출이 가장 높았던 3일을 뽑아보세요.
#        힌트: isin 으로 주말을 고르고 nlargest


# 문제 2. 공휴일의 평균 방문객과, 공휴일이 아닌 날의 평균 방문객을 비교해보세요.
#        힌트: df["is_holiday"] 는 이미 True/False 입니다


# 문제 3. 기온이 30도를 넘은 날은 며칠이고, 그날들의 평균 매출은 얼마인가요?

**모범 답안**

In [ ]:
# 문제 1
주말 = df[df["weekday"].isin(["토", "일"])]
print(주말.nlargest(3, "sales")[["date", "weekday", "sales"]])

# 문제 2
공휴일 = df[df["is_holiday"]]["visitors"].mean()
평일 = df[~df["is_holiday"]]["visitors"].mean()
print(f"\n공휴일 {공휴일:.1f}명 / 그 외 {평일:.1f}명 → {공휴일 - 평일:+.1f}명")

# 문제 3
무더위 = df[df["avg_temp"] > 30]
print(f"\n30도 초과: {len(무더위)}일, 평균 매출 {무더위['sales'].mean():,.0f}원")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)